# TuneBench training on a free cloud GPU

A thin wrapper: all the logic is in `src/tunebench/`, tested in the repo. This notebook only clones the repo, installs the locked dependencies and calls the training script.

On **Kaggle**: Settings → Accelerator → GPU T4, and turn Internet on. On **Colab**: Runtime → Change runtime type → T4 GPU. The T4 has no bfloat16, so training uses float16 there (picked automatically).

The results in the repo were trained on an Apple M5 Pro GPU; a T4 gives the same recipe, not identical numbers.

In [ ]:
!git clone --depth 1 https://github.com/olioli9586/tunebench
%cd tunebench
!pip install -q uv==0.13.7
!uv sync --locked --no-default-groups --group analysis --group train

Pick a recipe. `modernbert` is a full fine-tune of ModernBERT-base (about 10 minutes on a T4 per seed); `qwen3-lora` trains LoRA adapters on Qwen3-0.6B (longer).

In [ ]:
MODEL = "modernbert"  # or "qwen3-lora"
SEED = 0
!uv run python -m tunebench.train --model {MODEL} --seed {SEED}

Scores and test predictions land in `results/<model>-seed<seed>.json`; the MLflow runs in `mlflow.db`.

In [ ]:
import json
import pathlib

run = json.loads(pathlib.Path(f"results/{MODEL}-seed{SEED}.json").read_text())
{k: run[k] for k in ("accuracy", "macro_f1", "best_epoch", "train_seconds", "device")}